# GROOPY - Fingerspelling CNN Bake-off
### CRISP-DM: Modeling + Evaluation

Train **4 image classifiers under one fixed protocol** on the Kaggle **ASL Alphabet** (87k images, 29 classes), then score them and pick a winner.

| Candidate | Type |
|---|---|
| `cnn_scratch` | CNN built from scratch (baseline) |
| `mobilenetv2` | ImageNet transfer (mobile-friendly) |
| `efficientnetb0` | ImageNet transfer (accuracy/size sweet-spot) |
| `resnet50` | ImageNet transfer (high-capacity reference) |

Winner = a **weighted scorecard** (accuracy + latency + size + robustness + stability), i.e. the most *shippable* model, not just the most accurate.

> On Colab set a GPU first: **Runtime -> Change runtime type -> T4 GPU**. (Runs locally too, but CPU training is slow - Colab recommended.)

## 1 - Setup
Colab: clone + download (legacy `kaggle.json`; accept the dataset terms once). Local: makes the repo importable (data already present).

In [ ]:
import os, sys
try:
    import google.colab            # ---- running on Colab ----
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Pin the SAME tensorflow version as the local .venv (see requirements.txt) so the saved
    # .keras files stay in one Keras 3.x line. Colab's default runtime drifts forward over time;
    # an unpinned install here is what caused local (Keras 2) to fail loading these models before.
    !pip install -q "tensorflow==2.17.1"
    !git clone https://github.com/SpliiiiT/GROOPY.git 2>/dev/null || (cd GROOPY && git pull -q)
    %cd /content/GROOPY
    sys.path.insert(0, '/content/GROOPY')
    !pip install -q kaggle
    from google.colab import files; files.upload()      # upload kaggle.json
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !python data/download_asl_alphabet.py
else:                               # ---- running locally ----
    sys.path.insert(0, os.path.abspath('../..'))   # repo root
    print('Local run - using data/asl_alphabet_train (already downloaded).')


## 2 - Confirm the GPU
Training on CPU is far too slow - make sure a T4 is attached (Colab).

In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF', tf.__version__, '| GPU:', gpus or 'NONE - set Runtime > Change runtime type > T4')

## 3 - Build a fast training subset
ASL Alphabet has 3000 imgs/class - very redundant. Decoding all 87k every epoch on Colab's 2 CPUs is the bottleneck (~15 min/epoch). We **symlink ~600/class** (instant, no extra disk) so epochs run in ~1-3 min while accuracy stays high. The bake-off stays fair - every candidate uses the same subset + protocol.

_More accuracy: raise `N` to ~1000 and `epochs` to ~12. Full run: point `data_dir` at the original folder (much slower)._

In [ ]:
import os, glob
SRC, DST, N = os.path.abspath('data/asl_alphabet_train'), os.path.abspath('asl_subset'), 600
for cls in sorted(os.listdir(SRC)):
    os.makedirs(f'{DST}/{cls}', exist_ok=True)
    for f in sorted(glob.glob(f'{SRC}/{cls}/*.jpg'))[:N]:
        link = f'{DST}/{cls}/{os.path.basename(f)}'
        if not os.path.exists(link):
            try:
                os.symlink(f, link)
            except OSError:
                import shutil; shutil.copy(f, link)   # Windows without symlink perms
print(f'subset ready: {N} imgs/class at {DST}')

## 4 - Train the bake-off
One fixed protocol for every model (same split, augmentation, batch size, epochs). Pretrained backbones add a 2-phase fine-tune. `EarlyStopping` trims plateaus.

In [ ]:
from recognition.src.train import train_one
from recognition.src.data import make_datasets
from recognition.src import models as zoo

train_ds, val_ds, test_ds, class_names = make_datasets(data_dir='asl_subset', batch_size=32)
records = [train_one(name, epochs=10, train_ds=train_ds, val_ds=val_ds)
           for name in zoo.REGISTRY]
records

## 5 - Evaluate + weighted scorecard -> winner
Scores every model on the held-out test set, then ranks by the scorecard (accuracy 40%, latency 20%, size 15%, robustness 15%, stability 10%).

In [ ]:
from recognition.src.evaluate import evaluate_model
from recognition.src import scorecard as sc
from recognition.src.config import MODELS_DIR
from pathlib import Path
import pandas as pd

rows = [evaluate_model(p.stem, p, test_ds, class_names)
        for p in sorted(Path(MODELS_DIR).glob('*.keras')) if not p.stem.startswith('word_')]
ranked = sc.score(rows)
pd.DataFrame(ranked)[['model', 'accuracy', 'macro_f1', 'latency', 'size', 'total']]

## 6 - Trust check + export the winner
**Grad-CAM** confirms each model looks at the *hand*, not the background - set each model's `robustness` from the heatmaps, add `stability` from the live desktop test, then re-run cell 5. Finally export the winner and download it from **Files** into your local `recognition/models/`.

In [ ]:
winner = ranked[0]['model']; print('winner:', winner)
!python -m recognition.src.xai_gradcam --model recognition/models/{winner}.keras --n 8
!python -m recognition.src.export --model recognition/models/{winner}.keras --target all

## 7 - Download everything you need back to local
Zips the training curves (`history_*.json`), the scorecard artifacts (`bakeoff.json`/`bakeoff.md`/
`winner.json`), and the two models you don't have locally yet (`mobilenetv2.keras`, `resnet50.keras`).
`cnn_scratch.keras`/`efficientnetb0.keras` are **not** included — you already have working, verified
copies locally; grab a fresh one manually from the **Files** panel only if you actually want to
replace them.

In [ ]:
import zipfile
from pathlib import Path
from recognition.src.config import RESULTS_DIR, MODELS_DIR

zip_path = Path('groopy_cnn_bakeoff.zip')
wanted = (
    list(RESULTS_DIR.glob('history_*.json'))
    + [RESULTS_DIR / f for f in ('bakeoff.json', 'bakeoff.md', 'winner.json')]
    + [MODELS_DIR / f for f in ('mobilenetv2.keras', 'resnet50.keras')]
)

with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in wanted:
        if f.exists():
            zf.write(f, arcname=f.name)
        else:
            print(f'skip (not found): {f.name}')

print(f'zipped {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)')

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print(f'Local run - file is already at {zip_path.resolve()}')